# NIRISS/SOSS: HAT-P-14 b (real data)

This tutorial reduces the NIRISS/SOSS transit of HAT-P-14 b from JWST program 1541
(observation 1) and extracts orders 1, 2 and 3. It is **not executed** when the documentation
is built: it downloads about 6 GB of `rateints` files and needs CRDS reference files
(see {doc}`../installation`). The reduction takes a few minutes on a laptop.

In [ ]:
import glob
import numpy as np
import matplotlib.pyplot as plt
import transitspectroscopy as ts

## Download the ramp-fitted integrations

`ts.jwst.download` (needs the `download` extra) saves the files to `JWSTdata/`.

In [ ]:
ts.jwst.download(pid=1541, obs_num='1', data_product='rateints')
files = sorted(glob.glob('JWSTdata/jw01541001001_04101_00001-seg*_nis_rateints.fits'))
print(files)

## Reduce

The JWST SOSS background step removes the zodiacal background, PASTASOSS provides traces and
wavelengths for the three orders, and a 15-pixel box aperture is extracted. We also ask for
optimal extraction with a polynomial profile, fitted jointly where the orders overlap.

In [ ]:
result = ts.soss.reduce_soss(files, aperture_radius=15, outputfolder='soss_hatp14',
                             optimal_methods=('polynomial',))
times = result['times']

## Median spectra and white-light curves

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 7))
for order, products in result['orders'].items():
    axes[0].plot(products['wavelength'], np.nanmedian(products['flux'], axis=0), label=f'order {order}')
    axes[1].plot((times - times[0]) * 24, products['white_light'], '.', ms=2, label=f'order {order}')
axes[0].set_xlabel('Wavelength (µm)'); axes[0].set_ylabel('Median flux (DN/s)'); axes[0].legend()
axes[1].set_xlabel('Hours since start'); axes[1].set_ylabel('Relative flux');

Saved products (FITS, NPZ and CSV per order) are in `soss_hatp14/`. See
{doc}`../guides/soss` for the processing steps and {doc}`../validation/soss-report` for
how this reduction compares with published results.